In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Mount Google Drive                                     ║
# ╚══════════════════════════════════════════════════════════════════╝
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — API Keys                                              ║
# ╚══════════════════════════════════════════════════════════════════╝

GROQ_API_KEY = input("GrowQ API Key: ")
NGROK_TOKEN = input("NGROK Token: ")
print("✅ Keys set")


✅ Keys set


In [8]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Install Dependencies                                  ║
# ╚══════════════════════════════════════════════════════════════════╝
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "groq", "fpdf2", "pillow",
    "streamlit", "pyngrok",
    "opencv-python-headless", "pandas"
], check=True)
print("✅ Dependencies installed")


✅ Dependencies installed


In [9]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Write app.py + inject API key                        ║
# ╚══════════════════════════════════════════════════════════════════╝
APP = r'''
# -*- coding: utf-8 -*-
"""
NeuroScan AI  —  Brain Tumor MRI Analysis System  (Merged v5)
Pipeline: Quality Validation → Multi-Model Fusion → Segmentation → GradCAM
        → Size/Shape/Mass → Overlap Metrics → Risk/Reliability → Report
"""

import os, warnings, re, base64, json, tempfile
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import streamlit as st
import pandas as pd
from pathlib import Path
from datetime import datetime
from fpdf import FPDF
from groq import Groq
from PIL import Image

# ════════════════════════════════════════════════════════════════
# PAGE CONFIG
# ════════════════════════════════════════════════════════════════
st.set_page_config(
    page_title="NeuroScan AI",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ════════════════════════════════════════════════════════════════
# THEME CSS
# ════════════════════════════════════════════════════════════════
st.markdown("""
<style>
[data-testid="stAppViewContainer"]{background:#0a0c12;color:#c9cdd6}
[data-testid="stHeader"]{background:#0d0f18;border-bottom:1px solid #1a1f2e}
[data-testid="stSidebar"]{background:#0d0f18!important;border-right:1px solid #1a1f2e}
.block-container{padding:1.5rem 2rem 2rem}
div[data-testid="metric-container"]{background:#0d0f18;border:1px solid #1a1f2e;border-radius:12px;padding:14px 18px}
div[data-testid="metric-container"] label{color:#3d6fa5!important;font-size:11px;text-transform:uppercase;letter-spacing:.1em}
div[data-testid="stMetricValue"]{color:#5ea8ff;font-size:22px;font-weight:700}
[data-testid="stFileUploadDropzone"]{background:#0b1220!important;border:1.5px dashed #1e3050!important;border-radius:12px!important}
.stButton>button{background:linear-gradient(135deg,#1a3a70,#0f2050)!important;color:#7ab8ff!important;border:1px solid #1e4080!important;border-radius:10px!important;font-weight:600!important;width:100%;transition:all .2s}
.stButton>button:hover{background:linear-gradient(135deg,#1f4585,#142860)!important;border-color:#4a9eff!important}
[data-testid="stDownloadButton"]>button{background:linear-gradient(135deg,#0f2a10,#082010)!important;color:#4dc86f!important;border:1px solid #1a4020!important;border-radius:10px!important;font-weight:600!important;width:100%}
.streamlit-expanderHeader{background:#0d0f18!important;color:#c9cdd6!important;border:1px solid #1a1f2e!important;border-radius:10px!important}
.streamlit-expanderContent{background:#0d0f18!important;border:1px solid #1a1f2e!important;border-radius:0 0 10px 10px!important}
hr{border-color:#1a1f2e!important}
[data-testid="stDataFrame"]{border:1px solid #1a1f2e;border-radius:12px}
[data-testid="stAlert"]{background:#0d0f18!important;border:1px solid #1a1f2e!important;border-radius:10px!important}
div[data-testid="stSuccess"]{border-left:3px solid #4dc86f!important}
div[data-testid="stError"]{border-left:3px solid #e8604a!important}
div[data-testid="stWarning"]{border-left:3px solid #d4a843!important}
div[data-testid="stInfo"]{border-left:3px solid #4a9eff!important}
section[data-testid="stSidebar"] p,section[data-testid="stSidebar"] label{color:#7a8299}
section[data-testid="stSidebar"] h1,section[data-testid="stSidebar"] h2,section[data-testid="stSidebar"] h3{color:#4a6fa5}
[data-testid="stImage"] img{border-radius:10px;border:1px solid #1a1f2e}
h1{color:#e2e5ed!important;font-weight:700!important}
h2,h3{color:#c8cfe0!important}
p,li{color:#8090aa}
.tag-red  {background:#2a1010;color:#e87060;border:1px solid #3a1818;padding:3px 10px;border-radius:5px;font-size:11px;font-weight:700}
.tag-amber{background:#221a08;color:#d4a843;border:1px solid #3a2c0a;padding:3px 10px;border-radius:5px;font-size:11px;font-weight:700}
.tag-green{background:#0f2a10;color:#4dc86f;border:1px solid #1a4020;padding:3px 10px;border-radius:5px;font-size:11px;font-weight:700}
.tag-blue {background:#0d1a2a;color:#4a9eff;border:1px solid #1a2e4a;padding:3px 10px;border-radius:5px;font-size:11px;font-weight:700}
</style>
""", unsafe_allow_html=True)

# ════════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════════
BASE_DIR      = Path("/content/drive/MyDrive/Project work")
MODELS_DIR    = BASE_DIR / "models"
REPORT_DIR    = BASE_DIR / "Reports"
HISTORY_FILE  = BASE_DIR / "case_history.json"
ENV_FILE      = BASE_DIR / "CODE" / ".env.txt"
SAMPLE_DIR    = BASE_DIR / "Test Data"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def load_env(path):
    data = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            if "=" in line:
                k, v = line.split("=", 1)
                data[k.strip()] = v.strip().strip('"').strip("'")
    return data

ENV = load_env(ENV_FILE)
GROQ_API_KEY = input("Groq API Key: ")

IMG_SIZE    = 384
SEG_SIZE    = 256
PIX_MM      = 0.5
MM2_CM2     = 0.01
CLASS_NAMES = ["glioma", "meningioma", "no_tumor", "pituitary"]
MRI_PARAMS  = {
    "Sequence"       : "T1-Weighted Contrast Enhanced",
    "Resolution"     : "384 x 384 px",
    "Slice Thickness": "5.0 mm",
    "Field Strength" : "1.5 T",
    "Voxel Size"     : "0.5 x 0.5 x 5.0 mm",
}

# Multi-model specs — weights reflect relative expected accuracy
MODEL_SPECS = {
    "EfficientNetV2-S": {
        "path"      : MODELS_DIR / "class_Tumor_v2s_clean.keras",
        "preprocess": tf.keras.applications.efficientnet_v2.preprocess_input,
        "weight"    : 1.2,
    },
    "MobileNetV3": {
        "path"      : MODELS_DIR / "class_Tumor_mobilenet_v3.keras",
        "preprocess": tf.keras.applications.mobilenet_v3.preprocess_input,
        "weight"    : 1.0,
    },
    "ConvNeXt Tiny": {
        "path"      : MODELS_DIR / "class_Tumor_convnext_tiny_tumor.keras",
        "preprocess": tf.keras.applications.convnext.preprocess_input,
        "weight"    : 1.1,
    },
}
SEG_MODEL_PATH = MODELS_DIR / "Segmentation_brisc_effunet.keras"

# ════════════════════════════════════════════════════════════════
# PDF-SAFE TEXT SANITISER
# ════════════════════════════════════════════════════════════════
def safe(text):
    if not isinstance(text, str): text = str(text)
    _map = {'\u2014':'-','\u2013':'-','\u2012':'-','\u2010':'-','\u2011':'-',
            '\u2018':"'",'\u2019':"'",'\u201c':'"','\u201d':'"',
            '\u2022':'-','\u2026':'...','\u00b2':'2','\u00b3':'3',
            '\u00b0':' deg','\u00b1':'+/-','\u2192':'->','\u2265':'>=',
            '\u2264':'<=','\u2260':'!=','\u00b5':'u','\u00d7':'x'}
    for c, r in _map.items(): text = text.replace(c, r)
    return text.encode('latin-1', errors='ignore').decode('latin-1')

def clean_text(text):
    text = re.sub(r'#{1,6}\s*', '', text)
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
    text = re.sub(r'\*(.*?)\*', r'\1', text)
    text = re.sub(r'`(.*?)`', r'\1', text)
    return safe(text)

# ════════════════════════════════════════════════════════════════
# MODEL LOADING  — multi-model + segmentation + GradCAM heads
# ════════════════════════════════════════════════════════════════
@st.cache_resource(show_spinner="Loading AI models...")
def load_models():
    classifiers = {}
    for name, spec in MODEL_SPECS.items():
        try:
            classifiers[name] = tf.keras.models.load_model(str(spec["path"]), compile=False)
        except Exception as e:
            st.warning(f"Could not load {name}: {e}")
    seg_model = tf.keras.models.load_model(str(SEG_MODEL_PATH), compile=False)
    # GradCAM components from EfficientNetV2-S (primary classifier)
    base_model, head_layers = None, []
    if "EfficientNetV2-S" in classifiers:
        cm = classifiers["EfficientNetV2-S"]
        try:
            base_model = cm.get_layer("efficientnetv2-s")
            hit = False
            for layer in cm.layers:
                if layer.name == "efficientnetv2-s": hit = True; continue
                if hit: head_layers.append(layer)
        except Exception:
            pass
    return classifiers, seg_model, base_model, head_layers

# ════════════════════════════════════════════════════════════════
# FEATURE 1 — SCAN QUALITY VALIDATION
# ════════════════════════════════════════════════════════════════
def quality_metrics(image_np):
    gray = image_np.mean(axis=2).astype(np.float32)
    gy, gx = np.gradient(gray)
    blur = float(np.var(np.abs(gx) + np.abs(gy)))
    brightness = float(np.mean(gray))
    contrast = float(np.std(gray))
    quality_score = (
        0.25
        + 0.30 * min(blur / 150.0, 1.0)
        + 0.20 * (1.0 - min(abs(brightness - 127.5) / 127.5, 1.0))
        + 0.25 * min(contrast / 64.0, 1.0)
    )
    return {
        "quality_score"     : round(quality_score, 4),
        "blur_metric"       : round(blur, 4),
        "brightness_metric" : round(brightness, 4),
        "contrast_metric"   : round(contrast, 4),
        "usable_for_analysis": bool(quality_score >= 0.45),
    }

# ════════════════════════════════════════════════════════════════
# FEATURE 2 — MULTI-MODEL FUSION
# ════════════════════════════════════════════════════════════════
def preprocess_for_model(image_pil, model_name):
    arr = np.array(image_pil.convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32)
    prep = MODEL_SPECS[model_name]["preprocess"]
    try:
        arr = prep(arr)
    except Exception:
        arr = arr / 255.0
    return np.expand_dims(arr, axis=0)

def normalize_scores(scores):
    arr = np.asarray(scores, dtype=np.float32)
    total = float(arr.sum())
    return arr / total if total > 0 else np.zeros_like(arr)

def classify_image(classifiers, image_pil):
    raw = {}
    pretty = {}
    for name, model in classifiers.items():
        x = preprocess_for_model(image_pil, name)
        probs = normalize_scores(model.predict(x, verbose=0)[0])
        raw[name] = probs
        pretty[name] = {label: round(float(s), 4) for label, s in zip(CLASS_NAMES, probs)}
    # Weighted fusion
    weighted = np.zeros(len(CLASS_NAMES), dtype=np.float32)
    total_w = 0.0
    votes = {}
    for name, probs in raw.items():
        wt = MODEL_SPECS[name]["weight"]
        weighted += probs * wt
        total_w += wt
        votes[name] = CLASS_NAMES[int(np.argmax(probs))]
    weighted /= max(total_w, 1.0)
    ranked = np.argsort(weighted)[::-1]
    top_idx = int(ranked[0])
    top = float(weighted[top_idx])
    second = float(weighted[int(ranked[1])]) if len(ranked) > 1 else 0.0
    agreement = sum(1 for v in votes.values() if v == CLASS_NAMES[top_idx]) / max(len(votes), 1)
    uncertainty_flag = bool((top - second) < 0.08)
    return {
        "final_class"      : CLASS_NAMES[top_idx],
        "fused_confidence" : round(top, 4),
        "agreement_score"  : round(float(agreement), 4),
        "uncertainty_flag" : uncertainty_flag,
        "decision_logic"   : "Escalate for review" if uncertainty_flag else "Accept fused output",
        "model_votes"      : votes,
        "class_scores"     : {n: round(float(s), 4) for n, s in zip(CLASS_NAMES, weighted)},
    }, pretty

# ════════════════════════════════════════════════════════════════
# SEGMENTATION + GRADCAM
# ════════════════════════════════════════════════════════════════
def run_segmentation(seg_model, tmp_path):
    try:
        in_shape = seg_model.input_shape
        seg_h = int(in_shape[1]) if in_shape[1] else SEG_SIZE
        seg_w = int(in_shape[2]) if in_shape[2] else SEG_SIZE
        seg_ch = int(in_shape[3]) if in_shape[3] else 1
    except Exception:
        seg_h, seg_w, seg_ch = SEG_SIZE, SEG_SIZE, 3
    img_bgr = cv2.imread(tmp_path)
    if seg_ch == 1:
        gray   = cv2.resize(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY), (seg_w, seg_h))
        seg_in = np.expand_dims(gray / 255.0, (0, -1))
    else:
        rgb    = cv2.resize(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), (seg_w, seg_h))
        seg_in = np.expand_dims(rgb.astype(np.float32) / 255.0, 0)
    raw = seg_model(seg_in, training=False)
    out = raw[0][0].numpy() if isinstance(raw, (list, tuple)) else raw[0].numpy()
    if out.ndim == 3: out = out[:, :, 0]
    mask = (out > 0.5).astype(np.uint8)
    return cv2.resize(mask, (IMG_SIZE, IMG_SIZE))

def get_gradcam(img_tensor, base_model, head_layers):
    if base_model is None: return np.zeros((12, 12))
    img_t = tf.cast(img_tensor, tf.float32)
    conv_out = base_model(img_t, training=False)
    conv_var  = tf.Variable(conv_out)
    with tf.GradientTape() as tape:
        tape.watch(conv_var)
        x = conv_var
        for lyr in head_layers: x = lyr(x)
        score = x[:, tf.argmax(x[0])]
    grads = tape.gradient(score, conv_var)
    if grads is None: return np.zeros((12, 12))
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    hmap   = conv_out[0] @ pooled[..., tf.newaxis]
    hmap   = tf.squeeze(hmap)
    hmap   = tf.maximum(hmap, 0) / (tf.math.reduce_max(hmap) + 1e-8)
    return hmap.numpy()

# ════════════════════════════════════════════════════════════════
# SIZE / SHAPE / MASS EFFECT
# ════════════════════════════════════════════════════════════════
def estimate_size(mask):
    tp   = int(np.sum(mask)); tot = mask.shape[0] * mask.shape[1]
    amm2 = tp * (PIX_MM**2); acm2 = amm2 * MM2_CM2
    dcm  = (2 * np.sqrt(amm2 / np.pi)) / 10 if tp else 0.0
    pct  = (tp / (tot * 0.60)) * 100; vol = acm2 * 0.5
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bbox = None
    if cnts:
        x, y, w, h = cv2.boundingRect(max(cnts, key=cv2.contourArea))
        bbox = {"x":x,"y":y,"w":w,"h":h,
                "width_cm":round(w*PIX_MM/10,2),"height_cm":round(h*PIX_MM/10,2)}
    return {"tumor_pixels":tp,"area_mm2":round(amm2,2),"area_cm2":round(acm2,4),
            "diameter_cm":round(dcm,3),"tumor_percent":round(pct,2),
            "volume_cm3":round(vol,3),"bbox":bbox}

def analyze_shape(mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return None
    lg = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(lg); peri = cv2.arcLength(lg, True)
    if area == 0 or peri == 0: return None
    irr  = min(round(1-(4*np.pi*area)/(peri**2),3),1.0)
    comp = round((peri**2)/(4*np.pi*area),3)
    hull = cv2.convexHull(lg)
    conv = round(area/cv2.contourArea(hull),3) if cv2.contourArea(hull)>0 else 1.0
    ecc  = 0.0
    if len(lg)>=5:
        el=cv2.fitEllipse(lg); a,b=el[1][0]/2,el[1][1]/2
        if max(a,b)>0: ecc=round(np.sqrt(1-(min(a,b)/max(a,b))**2),3)
    bdef  = "Poorly defined" if irr>0.5 else ("Moderately defined" if irr>0.3 else "Well-defined")
    rough = "High" if irr>0.5 else ("Moderate" if irr>0.3 else "Low")
    return {"irregularity":irr,"compactness":comp,"convexity":conv,
            "eccentricity":ecc,"border_def":bdef,"roughness":rough}

def mass_effect(mask):
    h,w=mask.shape; mid=w//2
    lt=int(np.sum(mask[:,:mid])); rt=int(np.sum(mask[:,mid:]))
    tot=lt+rt; lat="None"; smm=0.0
    if tot>0:
        r=lt/tot
        lat="Left hemisphere" if r>0.6 else("Right hemisphere" if r<0.4 else "Bilateral/Midline")
        smm=round(abs(lt-rt)*(PIX_MM**2)/(h*PIX_MM),2)
    comp="Moderate" if smm>5 else("Mild" if smm>2 else "None")
    return {"laterality":lat,"shift_mm":smm,"compression":comp,
            "sulcal":"Present" if smm>2 else "Absent"}

# ════════════════════════════════════════════════════════════════
# FEATURE 7 — OVERLAP / EXPLAINABILITY METRICS
# ════════════════════════════════════════════════════════════════
def overlap_metrics(heatmap, seg_mask, heat_threshold=0.60):
    heat_region = (heatmap >= heat_threshold).astype(np.uint8)
    lesion      = (seg_mask > 0).astype(np.uint8)
    inter = int(np.logical_and(heat_region, lesion).sum())
    union = int(np.logical_or(heat_region, lesion).sum())
    hp    = int(heat_region.sum())
    iou     = inter / union if union else 0.0
    inside  = inter / hp if hp else 0.0
    return {
        "overlap_score"                    : round(float(iou), 4),
        "attention_inside_lesion_percent"  : round(float(inside * 100), 2),
        "explainability_consistency"       : round(float((iou + inside) / 2), 4),
    }

# ════════════════════════════════════════════════════════════════
# FEATURE 8/9 — RELIABILITY + RISK SCORE
# ════════════════════════════════════════════════════════════════
def reliability_and_risk(label, confidence, agreement, quality_score,
                          size_info, shape_info, mass_info, overlap_score):
    if label == "no_tumor":
        reliability = (0.35*confidence + 0.25*agreement
                       + 0.20*quality_score + 0.20*overlap_score)
        return {"severity":"None","risk":"None","clinical_priority":"Routine",
                "reliability_score":round(float(reliability),4),
                "progression_risk":"0%","score":0.0}
    sh_irr = shape_info["irregularity"] if shape_info else 0.0
    severity_index = (
        min(size_info["area_cm2"] / 25.0, 1.0) * 0.25
        + min(size_info["diameter_cm"] / 6.0, 1.0) * 0.15
        + min(size_info["tumor_percent"] / 30.0, 1.0) * 0.15
        + min(size_info["volume_cm3"] / 20.0, 1.0) * 0.10
        + min(sh_irr, 1.0) * 0.15
        + min(mass_info["shift_mm"] / 10.0, 1.0) * 0.10
        + min(overlap_score, 1.0) * 0.10
    )
    reliability = (0.30*confidence + 0.25*agreement
                   + 0.20*quality_score + 0.25*overlap_score)
    severity = "Severe" if severity_index>=0.70 else("Moderate" if severity_index>=0.40 else "Mild")
    risk     = "High"   if severity_index>=0.75 else("Moderate" if severity_index>=0.45 else "Low")
    priority = "Urgent" if severity_index>=0.75 else("Priority" if severity_index>=0.45 else "Routine")
    prog     = f"{int(severity_index*100)}%" if severity_index>=0.7 else(
               f"{int(severity_index*80)}%"  if severity_index>=0.4 else
               f"{int(severity_index*60)}%")
    return {"severity":severity,"risk":risk,"clinical_priority":priority,
            "reliability_score":round(float(reliability),4),
            "progression_risk":prog,"score":round(float(severity_index),3)}

def confidence_cal(confidence):
    u = round((1-confidence)*0.5+0.01,3)
    return {"uncertainty":f"+/-{round(u*100,1)}%","cal_score":round(1-u,3)}

def get_severity_label(label, confidence, tumor_percent):
    if label=="no_tumor": return "None"
    if tumor_percent>20 or confidence>0.95: return "Severe"
    if tumor_percent>10 or confidence>0.80: return "Moderate"
    return "Mild"

def clinical_decision(label, severity, mass_info):
    if label=="no_tumor":
        return {"steps":["Routine follow-up MRI in 12 months",
                         "No immediate clinical action required",
                         "Maintain regular neurological check-ups"],"urgency":"LOW"}
    steps=["Neurosurgical consultation","Contrast MRI follow-up"]
    if severity=="Severe":
        steps+=["MR Spectroscopy","Perfusion MRI","Surgical planning","Biopsy / Histopathology"]; urg="CRITICAL"
    elif severity=="Moderate":
        steps+=["MR Spectroscopy","Neuropsychological assessment"]; urg="HIGH"
    else:
        steps+=["PET scan consideration","Neurological evaluation"]; urg="MODERATE"
    if mass_info and mass_info["shift_mm"]>3:
        steps.append("Immediate review for mass effect")
    return {"steps":steps,"urgency":urg}

def rano_assessment(label, size_info, shape_info):
    if label=="no_tumor": return None
    sc  ="Large" if size_info["diameter_cm"]>3 else("Medium" if size_info["diameter_cm"]>1.5 else "Small")
    nec ="Likely present" if(shape_info and shape_info["irregularity"]>0.5) else "Not clearly identified"
    enh ="Heterogeneous" if(shape_info and shape_info["irregularity"]>0.4) else "Homogeneous"
    grd ="High-Grade (imaging features suggest possible high-grade - histopathological confirmation required)"
    if size_info["diameter_cm"]<2 and(not shape_info or shape_info["irregularity"]<0.3):
        grd="Low-Grade (imaging features suggest possible low-grade - histopathological confirmation required)"
    return {"size_cat":sc,"enhancement":enh,"necrosis":nec,"grade":grd}

# ════════════════════════════════════════════════════════════════
# FEATURE 10 — PRIOR-CASE COMPARISON
# ════════════════════════════════════════════════════════════════
def compare_with_prior(history, filename, current_area):
    matches = [row for row in history if row.get("filename") == filename]
    prior_area = None
    for row in reversed(matches):
        area = row.get("area_cm2")
        if isinstance(area, (int, float)):
            prior_area = float(area); break
    if prior_area is None or prior_area <= 0:
        return {"prior_available":False,"change_percent":None,"progression_flag":"Unknown"}
    change = ((current_area - prior_area) / prior_area) * 100.0
    return {"prior_available":True,"change_percent":round(float(change),2),
            "progression_flag":"Progression" if change>=20 else("Regression" if change<=-20 else "Stable")}

# ════════════════════════════════════════════════════════════════
# GROQ — TUMOR REPORT (full structured, with image)
# ════════════════════════════════════════════════════════════════
def groq_tumor_report(image_path, label, confidence, size_info,
                       shape_info, mass_info, risk_info, severity, rano, patient_name=""):
    client = Groq(api_key=GROQ_API_KEY)
    with open(image_path,"rb") as f: img_b64=base64.b64encode(f.read()).decode()
    bbox_t = (f"{size_info['bbox']['width_cm']} x {size_info['bbox']['height_cm']} cm"
              if size_info.get("bbox") else "N/A")
    rano_t = ("" if not rano else
              f"\n- RANO Size: {rano['size_cat']}\n- Enhancement: {rano['enhancement']}\n- Necrosis: {rano['necrosis']}")
    shape_t= ("" if not shape_info else
              f"\n- Irregularity: {shape_info['irregularity']}\n- Border: {shape_info['border_def']}")
    pt_line= f"\nPatient Name: {patient_name}" if patient_name else ""
    prompt = f"""You are a senior neuroradiology AI assistant generating a formal radiology report.
{pt_line}
AI ANALYSIS FINDINGS:
- Tumor Type: {label.upper()} | Confidence: {confidence:.2%} | Severity: {severity}
- Tumor Area: {size_info['area_cm2']} cm2 | Diameter: {size_info['diameter_cm']} cm | Est. Volume: {size_info['volume_cm3']} cm3
- Brain Coverage: {size_info['tumor_percent']}% | Bounding Box: {bbox_t}
- Laterality: {mass_info['laterality']} | Midline Shift: {mass_info['shift_mm']} mm
- Risk Score: {risk_info['score']}/1.0 | Growth Risk: {risk_info['risk']}{shape_t}{rano_t}

Generate a structured clinical radiology report with EXACTLY these 6 numbered sections.
Write each section as 2-4 complete professional sentences.

1. CLINICAL INDICATION
2. IMAGING TECHNIQUE
3. FINDINGS
4. IMPRESSION
5. SEVERITY ASSESSMENT
6. RECOMMENDATIONS

STRICT OUTPUT RULES:
- Use plain ASCII text only. No markdown, no bold, no bullets, no special characters.
- Never definitively confirm tumor grade — always say imaging features suggest possible grade only.
- Always state that histopathological confirmation is required for definitive diagnosis.
- Total report must be under 350 words.
- Start each section with its number and title on its own line."""
    resp = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role":"user","content":[
            {"type":"image_url","image_url":{"url":f"data:image/jpeg;base64,{img_b64}"}},
            {"type":"text","text":prompt}
        ]}], max_tokens=1200)
    return clean_text(resp.choices[0].message.content)

# ════════════════════════════════════════════════════════════════
# GROQ — NO TUMOR REPORT
# ════════════════════════════════════════════════════════════════
def groq_normal_report(image_path, confidence, patient_name=""):
    client = Groq(api_key=GROQ_API_KEY)
    with open(image_path,"rb") as f: img_b64=base64.b64encode(f.read()).decode()
    pt_line= f"\nPatient Name: {patient_name}" if patient_name else ""
    prompt = f"""You are a senior neuroradiology AI assistant generating a formal radiology report.
{pt_line}
AI CLASSIFICATION RESULT:
- Result: NO TUMOR DETECTED
- Classifier Confidence: {confidence:.2%}
- Severity: NONE

Generate a structured clinical radiology report with EXACTLY these 6 numbered sections.
Each section must reflect that NO tumor was found. Write 2-3 complete professional sentences per section.

1. CLINICAL INDICATION
2. IMAGING TECHNIQUE
3. FINDINGS
4. IMPRESSION
5. SEVERITY ASSESSMENT
6. RECOMMENDATIONS

STRICT OUTPUT RULES:
- Use plain ASCII text only. No markdown, no bold, no bullets, no special characters.
- Total report must be under 300 words.
- Start each section with its number and title on its own line."""
    resp = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role":"user","content":[
            {"type":"image_url","image_url":{"url":f"data:image/jpeg;base64,{img_b64}"}},
            {"type":"text","text":prompt}
        ]}], max_tokens=900)
    return clean_text(resp.choices[0].message.content)

# ════════════════════════════════════════════════════════════════
# RISK CHART
# ════════════════════════════════════════════════════════════════
def make_risk_chart(size_info, risk_info, shape_info, mass_info):
    DARK='#14284A'; RED='#C0392B'; ORG='#E67E22'; GRN='#27AE60'
    BLU='#2980B9'; PUR='#8E44AD'; TEA='#16A085'; BG='#F5F7FA'
    def rc(v): return RED if v>0.6 else(ORG if v>0.3 else GRN)
    fig, axes = plt.subplots(1,3,figsize=(14,3.2)); fig.patch.set_facecolor('#FFFFFF')
    tp=max(min(size_info['tumor_percent'],100),0)
    ax=axes[0]; ax.set_facecolor('#FFFFFF')
    _,_,ats=ax.pie([tp,100-tp],colors=[RED,GRN],autopct='%1.1f%%',startangle=90,
        pctdistance=0.78,wedgeprops=dict(width=0.55,edgecolor='white',linewidth=2),
        textprops={'fontsize':7.5,'color':'#333333'})
    for at in ats: at.set_fontsize(7); at.set_fontweight('bold'); at.set_color('white')
    ax.text(0,0,f"{tp:.1f}%\nTumor",ha='center',va='center',fontsize=8,fontweight='bold',color=DARK)
    ax.set_title('Tumor vs Brain Coverage',fontsize=9,fontweight='bold',color=DARK,pad=8)
    ax.legend(handles=[mpatches.Patch(facecolor=RED,label='Tumor'),
                        mpatches.Patch(facecolor=GRN,label='Healthy Brain')],
               fontsize=6.5,loc='lower center',bbox_to_anchor=(0.5,-0.12),ncol=2,frameon=False)
    ax=axes[1]; ax.set_facecolor(BG)
    for sp in ax.spines.values(): sp.set_color('#DDDDDD')
    cats=['Mass Effect','Shape','Size','Confidence']
    vals=[round(min(mass_info['shift_mm']/10,1.0),2),
          round(shape_info['irregularity'] if shape_info else 0,2),
          round(min(size_info['tumor_percent']/30,1.0),2),
          round(min(risk_info['score']*1.2,1.0),2)]
    bars=ax.barh(cats,vals,color=[rc(v) for v in vals],height=0.45,edgecolor='white',linewidth=0.8)
    ax.set_xlim(0,1.0); ax.set_xlabel('Score (0-1)',fontsize=7,color='#555555')
    ax.tick_params(axis='y',labelsize=7.5); ax.tick_params(axis='x',labelsize=6.5)
    ax.set_title('Risk Factor Analysis',fontsize=9,fontweight='bold',color=DARK,pad=8)
    for x in [0.3,0.6]: ax.axvline(x,color='#CCCCCC',linewidth=0.8,linestyle='--')
    for bar,v in zip(bars,vals):
        ax.text(min(v+0.03,0.97),bar.get_y()+bar.get_height()/2,f'{v:.2f}',
                va='center',ha='left',fontsize=7,fontweight='bold',color='#333333')
    ax.legend(handles=[mpatches.Patch(facecolor=RED,label='High > 0.6'),
                        mpatches.Patch(facecolor=ORG,label='Mod  > 0.3'),
                        mpatches.Patch(facecolor=GRN,label='Low <= 0.3')],
               fontsize=6,loc='lower right',frameon=True,framealpha=0.85)
    ax=axes[2]; ax.set_facecolor(BG)
    for sp in ax.spines.values(): sp.set_color('#DDDDDD')
    if shape_info:
        lbls=['Irregularity','Compactness\n(norm)','Eccentricity','Inv.Convexity']
        vals2=[shape_info['irregularity'],min(shape_info['compactness']/3.5,1.0),
               shape_info['eccentricity'],round(1-shape_info['convexity'],3)]
        b2=ax.bar(range(len(lbls)),vals2,color=[PUR,BLU,TEA,ORG],width=0.55,edgecolor='white',linewidth=0.8)
        ax.set_xticks(range(len(lbls))); ax.set_xticklabels(lbls,fontsize=6.2)
        ax.set_ylim(0,1.1); ax.set_ylabel('Score',fontsize=7,color='#555555')
        ax.tick_params(axis='y',labelsize=6.5)
        ax.axhline(0.5,color='#AAAAAA',linewidth=0.8,linestyle='--',label='Threshold')
        ax.legend(fontsize=6,loc='upper right',frameon=True,framealpha=0.85)
        for bar,v in zip(b2,vals2):
            ax.text(bar.get_x()+bar.get_width()/2,v+0.025,f'{v:.2f}',
                    ha='center',fontsize=7,fontweight='bold',color='#333333')
    else:
        ax.text(0.5,0.5,'No tumor data',ha='center',va='center',
                fontsize=10,color='#888888',transform=ax.transAxes)
    ax.set_title('Tumor Shape Metrics',fontsize=9,fontweight='bold',color=DARK,pad=8)
    fig.suptitle('Tumor Risk Analytics',fontsize=11,fontweight='bold',color=DARK,y=1.02)
    plt.tight_layout()
    os.makedirs('/tmp/rpt',exist_ok=True)
    p='/tmp/rpt/chart.jpg'
    plt.savefig(p,dpi=160,bbox_inches='tight',facecolor='white'); plt.close()
    return p

# ════════════════════════════════════════════════════════════════
# PDF HELPERS
# ════════════════════════════════════════════════════════════════
NAV=(20,40,74); TXT=(30,35,45); LGY=(200,205,210)
MUT=(100,110,125); BGC=(245,247,250); WHT=(255,255,255)

def _header(pdf, title, sub1, sub2=""):
    pdf.set_fill_color(*NAV); pdf.rect(0,0,210,14,'F')
    pdf.set_font("Helvetica","B",13); pdf.set_text_color(*WHT)
    pdf.set_xy(10,2); pdf.cell(125,10,safe(title))
    pdf.set_font("Helvetica","",7); pdf.set_text_color(180,200,230)
    pdf.set_xy(135,3); pdf.cell(65,4,safe(sub1),align="R")
    pdf.set_xy(135,8); pdf.cell(65,4,safe(sub2),align="R")
    pdf.set_text_color(*TXT)

def _footer(pdf, pg, total=3):
    pdf.set_xy(10,287); pdf.set_font("Helvetica","I",6.5); pdf.set_text_color(*MUT)
    pdf.cell(155,4,"AI-generated report. Must be reviewed by a qualified medical professional before clinical use.")
    pdf.set_font("Helvetica","B",7); pdf.set_xy(175,287)
    pdf.cell(25,4,f"Page {pg} / {total}",align="R"); pdf.set_text_color(*TXT)

def _secbar(pdf, title, y):
    pdf.set_fill_color(*NAV); pdf.rect(10,y,190,6,'F')
    pdf.set_font("Helvetica","B",7.5); pdf.set_text_color(*WHT)
    pdf.set_xy(13,y+0.9); pdf.cell(184,4.5,safe(f"  {title}"))
    pdf.set_text_color(*TXT); return y+7

def _card(pdf, x,y,w,h):
    pdf.set_fill_color(*BGC); pdf.set_draw_color(*LGY); pdf.rect(x,y,w,h,'FD')

def _divider(pdf, y):
    pdf.set_draw_color(*LGY); pdf.line(10,y,200,y)

def _kv(pdf, x,y,k,v,kw=32,vw=36,bold_val=False,vc=None):
    pdf.set_font("Helvetica","B",7); pdf.set_text_color(*MUT)
    pdf.set_xy(x,y); pdf.cell(kw,4.5,safe(f"{k}:"),ln=False)
    pdf.set_font("Helvetica","B" if bold_val else "",7.5)
    pdf.set_text_color(*(vc if vc else TXT))
    pdf.set_xy(x+kw,y); pdf.cell(vw,4.5,safe(str(v))); pdf.set_text_color(*TXT)

def _badge(pdf, x,y,text,color):
    tw=max(len(text)*2.0+6,18)
    pdf.set_fill_color(*color); pdf.rect(x,y,tw,5.5,'F')
    pdf.set_font("Helvetica","B",7.5); pdf.set_text_color(*WHT)
    pdf.set_xy(x,y); pdf.cell(tw,5.5,safe(text),align="C"); pdf.set_text_color(*TXT)

def _pt_banner(pdf, patient_name, patient_id="", y_after_header=14):
    if not patient_name: return y_after_header
    pdf.set_fill_color(12,22,50); pdf.rect(0,y_after_header,210,6,'F')
    pdf.set_font("Helvetica","",7.5); pdf.set_text_color(150,180,230)
    pdf.set_xy(10,y_after_header+1.2)
    id_part = f"   ID: {patient_id}" if patient_id else ""
    pdf.cell(190,4,safe(f"  Patient: {patient_name}{id_part}   |   Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"))
    pdf.set_text_color(*TXT); return y_after_header+6

def _llm_page(pdf, llm_report, title, sub1, sub2, pg, total, patient_name="", patient_id=""):
    pdf.add_page()
    _header(pdf, title, sub1, sub2)
    y=_pt_banner(pdf, patient_name, patient_id)
    y=max(y,14)+2
    _divider(pdf,y); y+=4
    for line in llm_report.split("\n"):
        line=safe(line.strip())
        if not line: y+=2; continue
        if y>278: break
        is_sec=any(line.startswith(h) for h in ["1.","2.","3.","4.","5.","6.","7.","8.","9."])
        if is_sec:
            y+=2
            pdf.set_fill_color(*NAV); pdf.rect(10,y,4,7,'F')
            pdf.set_fill_color(228,234,252); pdf.rect(14,y,186,7,'F')
            pdf.set_font("Helvetica","B",8.5); pdf.set_text_color(*NAV)
            pdf.set_xy(16,y+0.8); pdf.cell(182,5.5,safe(f" {line}"))
            pdf.set_text_color(*TXT); y+=9.5
        else:
            pdf.set_font("Helvetica","",8); pdf.set_text_color(45,50,60)
            pdf.set_xy(14,y); pdf.multi_cell(184,4.8,line)
            y=pdf.get_y()+1.5
    _footer(pdf,pg,total)

# ════════════════════════════════════════════════════════════════
# PDF — TUMOR  (3 pages) — now includes reliability + overlap + fusion
# ════════════════════════════════════════════════════════════════
def pdf_tumor(filename, image_path, label, confidence,
              size_info, shape_info, mass_info, risk_info, cal_info,
              severity, clinical, rano, llm_report,
              orig, mask_overlay, heatmap, gcam_overlay,
              patient_name="", patient_id="",
              fusion=None, quality=None, overlap=None, comparison=None):

    SC=(180,30,30) if severity in("Severe","Moderate") else (20,110,50)
    pdf=FPDF(); pdf.set_auto_page_break(auto=False)

    # ── PAGE 1: Clinical Analysis ─────────────────────────────
    pdf.add_page()
    _header(pdf,"AI Brain Tumor Analysis Report",
            f"File: {os.path.basename(filename)}",
            f"Sequence: {MRI_PARAMS['Sequence']}")
    y=_pt_banner(pdf,patient_name,patient_id); y=max(y,14)

    # Severity banner
    sc_color=(140,0,0) if severity=="Severe" else((155,75,0) if severity=="Moderate" else (0,110,50))
    pdf.set_fill_color(*sc_color); pdf.rect(10,y,190,8,'F')
    pdf.set_font("Helvetica","B",8); pdf.set_text_color(*WHT)
    banner=(f"  SEVERITY: {severity.upper()}   |   PREDICTION: {label.upper()}"
            f"   |   CONFIDENCE: {confidence:.2%}   |   GROWTH RISK: {risk_info['risk'].upper()}")
    pdf.set_xy(10,y+2); pdf.cell(190,4,safe(banner),align="C")
    pdf.set_text_color(*TXT); y+=10

    # Analysis Summary
    y=_secbar(pdf,"ANALYSIS SUMMARY",y)
    _card(pdf,10,y,190,38); ROW=7.2
    c1=[(12,"Tumor Type",label.upper()),(12,"Confidence",f"{confidence:.2%}"),
        (12,"Uncertainty",cal_info['uncertainty']),(12,"Calibration",str(cal_info['cal_score'])),
        (12,"Severity",severity)]
    c2=[(76,"Tumor Area",f"{size_info['area_cm2']} cm2"),(76,"Diameter",f"{size_info['diameter_cm']} cm"),
        (76,"Est. Volume",f"{size_info['volume_cm3']} cm3"),(76,"Brain Coverage",f"{size_info['tumor_percent']}%"),
        (76,"Bounding Box",(f"{size_info['bbox']['width_cm']} x {size_info['bbox']['height_cm']} cm")
         if size_info.get('bbox') else "N/A")]
    c3=[(140,"Laterality",mass_info['laterality']),(140,"Midline Shift",f"{mass_info['shift_mm']} mm"),
        (140,"Compression",mass_info['compression']),(140,"Sulcal Eff.",mass_info['sulcal']),
        (140,"Aggressiveness",f"{risk_info['score']:.3f} / 1.0")]
    for col in [c1,c2,c3]:
        for i,(cx,k,v) in enumerate(col):
            _kv(pdf,cx,y+2+i*ROW,k,v,kw=30,vw=34)
    y+=40

    # Multi-model fusion summary strip
    if fusion:
        y=_secbar(pdf,"MULTI-MODEL FUSION RESULT",y)
        FH=20; _card(pdf,10,y,190,FH)
        _kv(pdf,12,y+2,"Fusion Decision",fusion["decision_logic"],kw=30,vw=50)
        _kv(pdf,12,y+10,"Agreement",f"{fusion['agreement_score']*100:.0f}%",kw=30,vw=20)
        fw=45
        for i,(mn,mv) in enumerate(fusion["model_votes"].items()):
            _kv(pdf,80+i*fw,y+2,mn,mv.replace("_"," ").title(),kw=24,vw=20)
        if quality:
            _kv(pdf,80,y+10,"Scan Quality",str(quality["quality_score"]),kw=26,vw=20)
            _kv(pdf,126,y+10,"Reliability",str(risk_info["reliability_score"]),kw=24,vw=18)
        if overlap:
            _kv(pdf,162,y+10,"Exp. Consistency",str(overlap["explainability_consistency"]),kw=30,vw=16)
        y+=FH+3

    # Visual Analysis
    y=_secbar(pdf,"VISUAL ANALYSIS",y)
    tmp='/tmp/rpt'; os.makedirs(tmp,exist_ok=True)
    def sv(arr,name,cm_=None):
        p=f"{tmp}/{name}.jpg"
        if cm_ is None: cv2.imwrite(p,arr)
        else: plt.imsave(p,arr,cmap=cm_)
        return p
    imgs=[sv(orig,'orig'),sv(mask_overlay,'mask'),
          sv(heatmap,'hmap',cm_='jet'),sv(gcam_overlay,'gcam')]
    lbls=["Original MRI","Segmentation","GradCAM Heatmap","GradCAM Overlay"]
    IW,IH=44,40; GAP=(190-4*IW)/5
    _card(pdf,10,y,190,IH+12)
    for i,(p,l) in enumerate(zip(imgs,lbls)):
        ix=10+GAP+i*(IW+GAP)
        pdf.image(p,x=ix,y=y+2,w=IW,h=IH)
        pdf.set_font("Helvetica","B",6.5); pdf.set_text_color(*MUT)
        pdf.set_xy(ix,y+IH+3); pdf.cell(IW,4,safe(l),align="C")
    pdf.set_text_color(*TXT); y+=IH+16

    # Shape | Mass | Risk
    y=_secbar(pdf,"SHAPE ANALYSIS   |   MASS EFFECT   |   RISK SCORE",y)
    SH=32; _card(pdf,10,y,190,SH); RH=5.8
    pdf.set_draw_color(*LGY)
    pdf.line(74,y+2,74,y+SH-2); pdf.line(138,y+2,138,y+SH-2)
    if shape_info:
        _kv(pdf,12,y+2,"Irregularity",str(shape_info['irregularity']),kw=24,vw=18)
        _kv(pdf,12,y+2+RH,"Compactness",str(shape_info['compactness']),kw=24,vw=18)
        _kv(pdf,12,y+2+RH*2,"Convexity",str(shape_info['convexity']),kw=24,vw=18)
        _kv(pdf,12,y+2+RH*3,"Border",shape_info['border_def'],kw=24,vw=18)
        _kv(pdf,12,y+2+RH*4,"Roughness",shape_info['roughness'],kw=24,vw=18)
    _kv(pdf,78,y+2,"Laterality",mass_info['laterality'],kw=26,vw=28)
    _kv(pdf,78,y+2+RH,"Shift",f"{mass_info['shift_mm']} mm",kw=26,vw=28)
    _kv(pdf,78,y+2+RH*2,"Compression",mass_info['compression'],kw=26,vw=28)
    _kv(pdf,78,y+2+RH*3,"Sulcal",mass_info['sulcal'],kw=26,vw=28)
    _kv(pdf,142,y+2,"Aggress. Score",f"{risk_info['score']:.3f} / 1.0",kw=28,vw=22,bold_val=True,vc=SC)
    _kv(pdf,142,y+2+RH,"Growth Risk",risk_info['risk'],kw=28,vw=22,bold_val=True,vc=SC)
    _kv(pdf,142,y+2+RH*2,"Progression",risk_info['progression_risk'],kw=28,vw=22)
    _kv(pdf,142,y+2+RH*3,"Uncertainty",cal_info['uncertainty'],kw=28,vw=22)
    y+=SH+3

    # Prior comparison row
    if comparison and comparison.get("prior_available"):
        y=_secbar(pdf,"PRIOR-CASE COMPARISON",y)
        CH=14; _card(pdf,10,y,190,CH)
        flag=comparison["progression_flag"]; pct=comparison["change_percent"]
        fc=(180,30,30) if flag=="Progression" else((0,110,50) if flag=="Regression" else (30,80,150))
        _kv(pdf,12,y+2,"Progression Flag",flag,kw=28,vw=24,bold_val=True,vc=fc)
        _kv(pdf,90,y+2,"Area Change",f"{pct:+.1f}%",kw=22,vw=20,bold_val=True,vc=fc)
        y+=CH+3

    # RANO
    if rano:
        y=_secbar(pdf,"RANO CRITERIA ASSESSMENT",y)
        RAN_H=22; _card(pdf,10,y,190,RAN_H)
        _kv(pdf,12,y+2,"Size Category",rano['size_cat'],kw=28,vw=28)
        _kv(pdf,12,y+11,"Enhancement",rano['enhancement'],kw=28,vw=28)
        _kv(pdf,80,y+2,"Necrosis",rano['necrosis'],kw=22,vw=44)
        pdf.set_xy(80,y+11); pdf.set_font("Helvetica","B",7); pdf.set_text_color(*MUT)
        pdf.cell(22,4.5,"Grade:",ln=False)
        pdf.set_font("Helvetica","B",7); pdf.set_text_color(*SC)
        pdf.multi_cell(88,4.5,safe(rano['grade'])); pdf.set_text_color(*TXT)
        y+=RAN_H+3

    # Clinical Decision
    y=_secbar(pdf,"CLINICAL DECISION SUPPORT",y)
    n=len(clinical['steps']); mid=(n+1)//2; ch=mid*5.8+14
    _card(pdf,10,y,190,ch)
    pdf.set_xy(12,y+3); pdf.set_font("Helvetica","B",7.5); pdf.set_text_color(*MUT)
    pdf.cell(28,5,"Urgency Level:",ln=False)
    _badge(pdf,42,y+3,f"  {clinical['urgency']}  ",SC)
    for i,step in enumerate(clinical['steps']):
        col=0 if i<mid else 1; row=i if i<mid else i-mid
        pdf.set_xy(14+col*96,y+12+row*5.8)
        pdf.set_font("Helvetica","",7.5); pdf.set_text_color(*TXT)
        pdf.cell(90,4.8,safe(f"  >>  {step}"))

    _footer(pdf,1,3)

    # ── PAGE 2: Risk Dashboard ────────────────────────────────
    chart=make_risk_chart(size_info,risk_info,shape_info,mass_info)
    pdf.add_page()
    _header(pdf,"Risk Analytics Dashboard",
            f"File: {os.path.basename(filename)}",
            f"Prediction: {label.upper()}  |  Confidence: {confidence:.2%}  |  Severity: {severity}")
    y=_pt_banner(pdf,patient_name,patient_id); y=max(y,14)+2
    pdf.set_font("Helvetica","I",7.5); pdf.set_text_color(*MUT)
    pdf.set_xy(10,y); pdf.cell(190,5,safe("Quantitative breakdown of tumor characteristics derived from segmentation and classification models."))
    pdf.set_text_color(*TXT); y+=8
    _card(pdf,10,y,190,96); pdf.image(chart,x=12,y=y+3,w=186,h=90); y+=100

    y=_secbar(pdf,"KEY METRICS AT A GLANCE",y)
    STRIP=24; _card(pdf,10,y,190,STRIP)
    metrics=[("Tumor Area",f"{size_info['area_cm2']} cm2"),
             ("Diameter",f"{size_info['diameter_cm']} cm"),
             ("Volume",f"{size_info['volume_cm3']} cm3"),
             ("Brain Cov.",f"{size_info['tumor_percent']}%"),
             ("Midline Shift",f"{mass_info['shift_mm']} mm"),
             ("Risk Score",f"{risk_info['score']:.3f} / 1.0")]
    cw=190/len(metrics)
    for i,(mk,mv) in enumerate(metrics):
        cx=10+i*cw
        if i>0: pdf.set_draw_color(*LGY); pdf.line(cx,y+3,cx,y+STRIP-3)
        pdf.set_font("Helvetica","B",6.5); pdf.set_text_color(*MUT)
        pdf.set_xy(cx,y+4); pdf.cell(cw,4,safe(mk),align="C")
        pdf.set_font("Helvetica","B",10); pdf.set_text_color(*NAV)
        pdf.set_xy(cx,y+10); pdf.cell(cw,7,safe(mv),align="C")
    pdf.set_text_color(*TXT); y+=STRIP+4

    if shape_info:
        y=_secbar(pdf,"SHAPE DETAIL",y)
        SH2=28; _card(pdf,10,y,190,SH2)
        shape_rows=[("Irregularity",shape_info['irregularity'],"Higher = more irregular boundary"),
                    ("Compactness",round(min(shape_info['compactness']/3.5,1.0),3),"Normalised compactness (0-1)"),
                    ("Eccentricity",shape_info['eccentricity'],"Elongation of fitted ellipse"),
                    ("Convexity",shape_info['convexity'],"How convex the tumor contour is"),
                    ("Border",shape_info['border_def'],"Border definition quality"),
                    ("Roughness",shape_info['roughness'],"Surface texture assessment")]
        scw=190/3
        for i,(sk,sv2,desc) in enumerate(shape_rows):
            ci=i%3; ri=i//3; cx2=10+ci*scw+3; cy2=y+3+ri*11
            pdf.set_font("Helvetica","B",7); pdf.set_text_color(*MUT)
            pdf.set_xy(cx2,cy2); pdf.cell(scw-3,4.5,safe(f"{sk}:"),ln=False)
            pdf.set_font("Helvetica","B",8); pdf.set_text_color(*NAV)
            pdf.set_xy(cx2+28,cy2); pdf.cell(28,4.5,safe(str(sv2)),ln=False)
            pdf.set_font("Helvetica","I",6); pdf.set_text_color(*MUT)
            pdf.set_xy(cx2,cy2+5.5); pdf.cell(scw-3,4,safe(desc))
        pdf.set_text_color(*TXT); y+=SH2+4

    y=max(y,248)
    pdf.set_fill_color(255,248,225); pdf.set_draw_color(230,175,60); pdf.rect(10,y,190,16,'FD')
    pdf.set_font("Helvetica","B",7.5); pdf.set_text_color(150,90,0)
    pdf.set_xy(13,y+2.5); pdf.cell(10,5,"(!)",ln=False)
    pdf.set_font("Helvetica","",7); pdf.set_xy(21,y+2.5)
    pdf.multi_cell(176,5,safe(
        "IMPORTANT: This AI-generated analysis is intended for screening and research purposes only. "
        "All findings must be reviewed and validated by a qualified radiologist or neurosurgeon before "
        "any clinical decisions are made. Histopathological confirmation is required for definitive diagnosis."))
    pdf.set_text_color(*TXT)
    _footer(pdf,2,3)

    # ── PAGE 3: LLM Radiology Report ─────────────────────────
    _llm_page(pdf, llm_report,
              "AI Generated Radiology Report",
              f"File: {os.path.basename(filename)}",
              "Note: AI-generated imaging findings. Histopathological confirmation required.",
              3, 3, patient_name, patient_id)

    ts=datetime.now().strftime('%Y%m%d%H%M%S')
    sn=os.path.splitext(os.path.basename(filename))[0]
    pt=f"_{patient_name.replace(' ','_')}" if patient_name else ""
    out=str(REPORT_DIR / f"{sn}{pt}_{ts}_mri_report.pdf")
    pdf.output(out); return out

# ════════════════════════════════════════════════════════════════
# PDF — NO TUMOR  (2 pages)
# ════════════════════════════════════════════════════════════════
def pdf_normal(filename, image_path, confidence, llm_report, orig,
               patient_name="", patient_id="", fusion=None, quality=None):
    GRN_C=(0,120,55); pdf=FPDF(); pdf.set_auto_page_break(auto=False)
    pdf.add_page()
    _header(pdf,"AI Brain MRI Analysis Report",
            f"File: {os.path.basename(filename)}",
            f"Sequence: {MRI_PARAMS['Sequence']}")
    y=_pt_banner(pdf,patient_name,patient_id); y=max(y,14)
    pdf.set_fill_color(*GRN_C); pdf.rect(10,y,190,8,'F')
    pdf.set_font("Helvetica","B",8); pdf.set_text_color(*WHT)
    pdf.set_xy(10,y+2)
    pdf.cell(190,4,safe(f"  RESULT: NORMAL   |   NO TUMOR DETECTED   |   CONFIDENCE: {confidence:.2%}   |   SEVERITY: NONE"),align="C")
    pdf.set_text_color(*TXT); y+=10

    # Fusion summary if available
    if fusion:
        y=_secbar(pdf,"MULTI-MODEL FUSION RESULT",y)
        FH=14; _card(pdf,10,y,190,FH)
        _kv(pdf,12,y+2,"Fused Confidence",f"{fusion['fused_confidence']*100:.1f}%",kw=28,vw=20)
        _kv(pdf,80,y+2,"Agreement",f"{fusion['agreement_score']*100:.0f}%",kw=20,vw=16)
        if quality:
            _kv(pdf,140,y+2,"Scan Quality",str(quality["quality_score"]),kw=26,vw=20)
        y+=FH+3

    y=_secbar(pdf,"ANALYSIS SUMMARY",y)
    _card(pdf,10,y,190,36)
    rows=[("Tumor Type","NO TUMOR DETECTED"),("Classification Confidence",f"{confidence:.2%}"),
          ("Severity","None"),("Growth Risk","None"),("Risk Score","0.000 / 1.0"),
          ("Recommended Action","Routine follow-up in 12 months")]
    for i,(k,v) in enumerate(rows):
        col=0 if i<3 else 1; row=i if i<3 else i-3
        cx=12+col*96
        is_highlight=(k=="Tumor Type")
        _kv(pdf,cx,y+2+row*11,k,v,kw=46,vw=46,bold_val=True,vc=GRN_C if is_highlight else TXT)
    y+=38

    y=_secbar(pdf,"ORIGINAL MRI SCAN",y)
    tmp='/tmp/rpt'; os.makedirs(tmp,exist_ok=True)
    orig_p=f"{tmp}/orig_normal.jpg"; cv2.imwrite(orig_p,orig)
    img_h=85; _card(pdf,10,y,190,img_h+8)
    pdf.image(orig_p,x=10+(190-100)/2,y=y+4,w=100,h=img_h)
    y+=img_h+12

    y=_secbar(pdf,"CLINICAL DECISION SUPPORT",y)
    cal={"steps":["Routine follow-up MRI in 12 months if clinically indicated",
                  "No neurosurgical or oncological intervention required",
                  "Correlate with clinical symptoms and patient history",
                  "Maintain regular neurological check-ups"],"urgency":"LOW"}
    n=len(cal['steps']); mid=(n+1)//2; ch=mid*5.8+14
    _card(pdf,10,y,190,ch)
    pdf.set_xy(12,y+3); pdf.set_font("Helvetica","B",7.5); pdf.set_text_color(*MUT)
    pdf.cell(28,5,"Urgency Level:",ln=False)
    _badge(pdf,42,y+3,"  LOW  ",GRN_C)
    for i,step in enumerate(cal['steps']):
        col=0 if i<mid else 1; row=i if i<mid else i-mid
        pdf.set_xy(14+col*96,y+12+row*5.8)
        pdf.set_font("Helvetica","",7.5); pdf.set_text_color(*TXT)
        pdf.cell(90,4.8,safe(f"  >>  {step}"))
    y+=ch+4

    y=max(y,248)
    pdf.set_fill_color(230,255,235); pdf.set_draw_color(60,180,80); pdf.rect(10,y,190,16,'FD')
    pdf.set_font("Helvetica","B",7.5); pdf.set_text_color(0,110,40)
    pdf.set_xy(13,y+2.5); pdf.cell(10,5,"(!)",ln=False)
    pdf.set_font("Helvetica","",7); pdf.set_xy(21,y+2.5)
    pdf.multi_cell(176,5,safe(
        "IMPORTANT: This AI-generated result is intended for screening purposes only. "
        "All findings must be reviewed by a qualified radiologist before any clinical decisions are made. "
        "If the patient has symptoms, further investigation by a medical professional is strongly advised."))
    pdf.set_text_color(*TXT)
    _footer(pdf,1,2)

    _llm_page(pdf, llm_report,
              "AI Generated Radiology Report",
              f"File: {os.path.basename(filename)}",
              "Note: AI-generated imaging findings - No tumor detected.",
              2, 2, patient_name, patient_id)

    ts=datetime.now().strftime('%Y%m%d%H%M%S')
    sn=os.path.splitext(os.path.basename(filename))[0]
    pt=f"_{patient_name.replace(' ','_')}" if patient_name else ""
    out=str(REPORT_DIR / f"{sn}{pt}_{ts}_normal_report.pdf")
    pdf.output(out); return out

# ════════════════════════════════════════════════════════════════
# PATIENT HISTORY
# ════════════════════════════════════════════════════════════════
def load_history():
    if HISTORY_FILE.exists():
        try: return json.loads(HISTORY_FILE.read_text(encoding="utf-8"))
        except: return []
    return []

def save_history(record):
    h=load_history(); h.insert(0,record)
    HISTORY_FILE.write_text(json.dumps(h[:100],indent=2),encoding="utf-8")

# ════════════════════════════════════════════════════════════════
# SIDEBAR
# ════════════════════════════════════════════════════════════════
with st.sidebar:
    st.markdown("## 🧠 NeuroScan AI")
    st.caption("Brain Tumor MRI Analysis System")
    st.markdown("---")
    page=st.radio("Navigate",["🏥 Dashboard","📋 Patient History","ℹ️ Model Info"],
                  label_visibility="collapsed")
    st.markdown("---")
    st.markdown("### AI Pipeline")
    st.markdown("""<div style='font-size:12px;line-height:2.2'>
    <span style='color:#4dc86f'>✓</span> Scan Quality Validation<br>
    <span style='color:#4dc86f'>✓</span> Multi-Model Fusion (3 models)<br>
    <span style='color:#4dc86f'>✓</span> Confidence Decision Logic<br>
    <span style='color:#4dc86f'>✓</span> EfficientNet U-Net Segmentation<br>
    <span style='color:#4dc86f'>✓</span> Grad-CAM Explainability<br>
    <span style='color:#4dc86f'>✓</span> Tumor Size &amp; Shape Analysis<br>
    <span style='color:#4dc86f'>✓</span> Overlap / Reliability Scoring<br>
    <span style='color:#4dc86f'>✓</span> Prior-Case Comparison<br>
    <span style='color:#4dc86f'>✓</span> Groq Llama-4 Radiology Report<br>
    <span style='color:#4dc86f'>✓</span> 3-Page PDF Export (Tumor)<br>
    <span style='color:#4dc86f'>✓</span> 2-Page PDF Export (Normal)
    </div>""",unsafe_allow_html=True)
    st.markdown("---")
    st.caption("⚠️ For research & screening only. Not for clinical diagnosis.")

# ════════════════════════════════════════════════════════════════
# PAGE: DASHBOARD
# ════════════════════════════════════════════════════════════════
if page=="🏥 Dashboard":
    st.title("🧠 NeuroScan AI — Doctor Dashboard")
    st.caption("Upload a brain MRI scan to run the full AI analysis pipeline.")
    st.markdown("---")
    col_L, col_R = st.columns([1,1.7],gap="large")

    with col_L:
        st.subheader("📤 Scan Upload")
        # Patient metadata
        pm1,pm2=st.columns(2)
        with pm1: patient_name=st.text_input("Patient Name",placeholder="e.g. John Doe",key="pname")
        with pm2: patient_id=st.text_input("Patient ID",placeholder="e.g. MRI-001",key="pid")

        uploaded=st.file_uploader("Drop MRI image here",
                                   type=["png","jpg","jpeg","bmp"],
                                   label_visibility="collapsed")

        # Sample image picker
        sample_names = []
        if SAMPLE_DIR.exists():
            sample_names = sorted([p.name for p in SAMPLE_DIR.glob("*.jpg")])
        sample_choice = st.selectbox("Or choose a sample image",
                                     ["None"] + sample_names,
                                     label_visibility="visible")

        image_pil = None; source_name = None; tmp_path = None
        if uploaded is not None:
            image_pil = Image.open(uploaded); source_name = uploaded.name
            suffix="."+uploaded.name.split(".")[-1]
            tf_=tempfile.NamedTemporaryFile(delete=False,suffix=suffix)
            tf_.write(uploaded.getvalue()); tf_.close(); tmp_path=tf_.name
        elif sample_choice != "None" and SAMPLE_DIR.exists():
            image_pil = Image.open(SAMPLE_DIR / sample_choice); source_name = sample_choice
            tmp_path = str(SAMPLE_DIR / sample_choice)

        if image_pil is not None:
            st.image(image_pil,caption="Input MRI",use_container_width=True)
        run_btn=st.button("▶  Run Full Analysis",disabled=image_pil is None)

        if image_pil is not None and run_btn:
            prog=st.progress(0); status=st.empty()

            # STEP 0: Quality check
            status.info("⚙️ Step 0 / 6 — Validating scan quality…")
            image_np = np.array(image_pil.convert("RGB"))
            quality = quality_metrics(image_np)
            if not quality["usable_for_analysis"]:
                st.warning(f"⚠️ Low quality scan (score={quality['quality_score']}). Results may be unreliable.")
            prog.progress(10)

            # STEP 1: Multi-model classification
            status.info("⚙️ Step 1 / 6 — Running multi-model fusion…")
            classifiers, seg_model, base_m, heads = load_models()
            fusion, model_scores = classify_image(classifiers, image_pil)
            label      = fusion["final_class"]
            confidence = fusion["fused_confidence"]
            if fusion["uncertainty_flag"]:
                st.warning(f"⚠️ Uncertain prediction — models disagree. Decision: {fusion['decision_logic']}")
            prog.progress(25)

            history = load_history()
            orig = cv2.resize(cv2.imread(tmp_path),(IMG_SIZE,IMG_SIZE))

            # ── NO TUMOR PATH ─────────────────────────────────
            if label=="no_tumor":
                status.info("🟢 No tumor detected — generating normal report…")
                llm=groq_normal_report(tmp_path,confidence,patient_name)
                pdf_path=pdf_normal(source_name,tmp_path,confidence,llm,orig,
                                     patient_name,patient_id,fusion,quality)
                prog.progress(100); status.success("✅ Analysis complete — Normal MRI!")
                st.session_state["result"]={
                    "no_tumor":True,"label":label,"confidence":confidence,
                    "severity":"None","filename":source_name,
                    "patient_name":patient_name,"patient_id":patient_id,
                    "llm":llm,"pdf_path":pdf_path,"orig":orig,
                    "fusion":fusion,"quality":quality,
                    "model_scores":model_scores,
                }
                save_history({"id":f"#MRI-{len(history)+1:03d}",
                              "filename":source_name,"patient":patient_name or "—",
                              "label":"No Tumor","confidence":f"{confidence*100:.1f}%",
                              "severity":"None","area_cm2":"N/A","risk":"None",
                              "reliability":quality["quality_score"],
                              "date":datetime.now().strftime("%b %d, %Y %H:%M")})

            # ── TUMOR PATH ────────────────────────────────────
            else:
                status.info("⚙️ Step 2 / 6 — Segmenting tumor region…")
                mask=run_segmentation(seg_model,tmp_path); prog.progress(40)

                status.info("⚙️ Step 3 / 6 — Generating Grad-CAM heatmap…")
                # Need EfficientNetV2-S tensor for GradCAM
                from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as effv2_prep
                ic = effv2_prep(np.array(image_pil.convert("RGB").resize((IMG_SIZE,IMG_SIZE)),dtype=np.float32))
                it = tf.convert_to_tensor(np.expand_dims(ic,0))
                heatmap=get_gradcam(it,base_m,heads); prog.progress(55)

                status.info("⚙️ Step 4 / 6 — Computing metrics & overlap…")
                s_info=estimate_size(mask); sh_info=analyze_shape(mask)
                m_info=mass_effect(mask)
                hmap_r=cv2.resize(heatmap,(IMG_SIZE,IMG_SIZE))
                overlap=overlap_metrics(hmap_r,mask)
                r_info=reliability_and_risk(label,confidence,
                                             fusion["agreement_score"],
                                             quality["quality_score"],
                                             s_info,sh_info,m_info,
                                             overlap["overlap_score"])
                c_info=confidence_cal(confidence)
                severity=r_info["severity"]
                clin=clinical_decision(label,severity,m_info)
                rano=rano_assessment(label,s_info,sh_info)
                comparison=compare_with_prior(history,source_name,s_info["area_cm2"])
                prog.progress(70)

                # Build overlays
                mask_vis=np.uint8(mask*255)
                mask_col=cv2.applyColorMap(mask_vis,cv2.COLORMAP_HOT)
                mask_ov=cv2.addWeighted(orig,0.7,mask_col,0.3,0)
                if s_info.get('bbox'):
                    b=s_info['bbox']
                    cv2.rectangle(mask_ov,(b['x'],b['y']),(b['x']+b['w'],b['y']+b['h']),(0,255,0),2)
                    cv2.putText(mask_ov,f"{s_info['diameter_cm']}cm",
                                (b['x'],b['y']-8),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)
                hmap_c=cv2.applyColorMap(np.uint8(255*hmap_r),cv2.COLORMAP_JET)
                gcam_ov=cv2.addWeighted(orig,0.6,hmap_c,0.4,0)

                status.info("⚙️ Step 5 / 6 — Generating LLM radiology report…")
                llm=groq_tumor_report(tmp_path,label,confidence,s_info,sh_info,
                                       m_info,r_info,severity,rano,patient_name)
                prog.progress(85)

                status.info("⚙️ Step 6 / 6 — Building PDF report…")
                pdf_path=pdf_tumor(source_name,tmp_path,label,confidence,
                                    s_info,sh_info,m_info,r_info,c_info,
                                    severity,clin,rano,llm,
                                    orig,mask_ov,hmap_r,gcam_ov,
                                    patient_name,patient_id,
                                    fusion,quality,overlap,comparison)
                prog.progress(100); status.success("✅ Analysis complete!")
                st.session_state["result"]={
                    "no_tumor":False,"label":label,"confidence":confidence,
                    "severity":severity,"filename":source_name,
                    "patient_name":patient_name,"patient_id":patient_id,
                    "size_info":s_info,"shape_info":sh_info,"mass_info":m_info,
                    "risk_info":r_info,"cal_info":c_info,"clinical":clin,"rano":rano,
                    "llm":llm,"pdf_path":pdf_path,
                    "orig":orig,"mask_ov":mask_ov,"hmap_r":hmap_r,"gcam_ov":gcam_ov,
                    "fusion":fusion,"quality":quality,"overlap":overlap,"comparison":comparison,
                    "model_scores":model_scores,
                }
                save_history({"id":f"#MRI-{len(history)+1:03d}",
                              "filename":source_name,"patient":patient_name or "—",
                              "label":label.capitalize(),"confidence":f"{confidence*100:.1f}%",
                              "severity":severity,"area_cm2":s_info['area_cm2'],
                              "risk":r_info['risk'],"reliability":r_info["reliability_score"],
                              "date":datetime.now().strftime("%b %d, %Y %H:%M")})

        # ── RESULT PANEL ─────────────────────────────────────
        if "result" in st.session_state:
            res=st.session_state["result"]
            st.markdown("---"); st.subheader("🔬 Detection Result")
            sev=res['severity']
            tag=("tag-red" if sev=="Severe" else
                 "tag-amber" if sev=="Moderate" else "tag-green")
            icon="🔴" if sev=="Severe" else("🟡" if sev=="Moderate" else "🟢")
            fusion_r=res.get("fusion",{})
            decision_txt=fusion_r.get("decision_logic","")
            pt_disp=(f"<div style='font-size:11px;color:#5a6a80;margin-top:2px'>"
                     f"Patient: {res.get('patient_name','—')} | ID: {res.get('patient_id','—')}</div>"
                     if res.get('patient_name') or res.get('patient_id') else "")
            st.markdown(f"""
            <div style='background:#0d0f18;border:1px solid #1a1f2e;border-radius:12px;
                        padding:14px 18px;margin-bottom:12px;display:flex;align-items:center;gap:14px'>
              <div style='font-size:30px'>{icon}</div>
              <div>
                <div style='font-size:18px;font-weight:700;color:#e2e5ed'>
                  {res['label'].replace("_"," ").capitalize()}</div>
                <div style='font-size:11px;color:#4a6a8a'>Multi-Model Fusion  |  Confidence: {res['confidence']*100:.1f}%</div>
                <div style='font-size:11px;color:#3a5a70'>{decision_txt}</div>
                {pt_disp}
              </div>
              <div style='margin-left:auto'><span class='{tag}'>{sev.upper()}</span></div>
            </div>""",unsafe_allow_html=True)

            # Quality metrics
            if res.get("quality"):
                q=res["quality"]
                qc=st.columns(3)
                qc[0].metric("Scan Quality",str(q["quality_score"]))
                qc[1].metric("Blur Score",str(q["blur_metric"]))
                qc[2].metric("Contrast",str(q["contrast_metric"]))

            if res["no_tumor"]:
                st.success("✅ No intracranial mass or tumor identified. Brain appears normal.")
                c1,c2=st.columns(2)
                with c1: st.metric("Confidence",f"{res['confidence']*100:.1f}%")
                with c2: st.metric("Severity","None")
            else:
                c1,c2=st.columns(2)
                with c1:
                    st.metric("Confidence",f"{res['confidence']*100:.1f}%")
                    st.metric("Tumor Area",f"{res['size_info']['area_cm2']} cm²")
                    st.metric("Risk Score",f"{res['risk_info']['score']:.3f} / 1.0")
                with c2:
                    st.metric("Diameter",f"{res['size_info']['diameter_cm']} cm")
                    st.metric("Volume",f"{res['size_info']['volume_cm3']} cm³")
                    st.metric("Reliability",str(res['risk_info']['reliability_score']))

                # Model votes
                if res.get("fusion"):
                    st.markdown("**Model Votes:**")
                    fv=res["fusion"]["model_votes"]
                    vote_cols=st.columns(len(fv))
                    for i,(mn,mv) in enumerate(fv.items()):
                        vote_cols[i].metric(mn,mv.replace("_"," ").title())

                # Prior comparison
                comp=res.get("comparison",{})
                if comp.get("prior_available"):
                    flag=comp["progression_flag"]; pct=comp["change_percent"]
                    col = "🔴" if flag=="Progression" else("🟢" if flag=="Regression" else "🟡")
                    st.info(f"{col} **Prior Case Found** — {flag} | Area change: {pct:+.1f}%")

            st.markdown("---"); st.subheader("📋 Clinical Decision")
            urg=(res['clinical']['urgency'] if not res['no_tumor'] else "LOW")
            steps=(res['clinical']['steps'] if not res['no_tumor']
                   else ["Routine follow-up in 12 months","No immediate action required"])
            urg_col=("#e8604a" if urg=="CRITICAL" else
                     "#d4a843" if urg in("HIGH","MODERATE") else "#4dc86f")
            st.markdown(f"**Urgency:** <span style='color:{urg_col};font-weight:700'>{urg}</span>",
                        unsafe_allow_html=True)
            for step in steps: st.markdown(f"→ {step}")

            st.markdown("---"); st.subheader("📄 PDF Report")
            pdf_p=res.get("pdf_path","")
            if pdf_p and os.path.exists(pdf_p):
                with open(pdf_p,"rb") as f:
                    pt_t=(f"_{res.get('patient_name','').replace(' ','_')}"
                          if res.get('patient_name') else "")
                    st.download_button("⬇  Download Full PDF Report",data=f,
                                       file_name=f"neuroscan{pt_t}_{res['filename']}_{datetime.now().strftime('%Y%m%d%H%M')}.pdf",
                                       mime="application/pdf",use_container_width=True)

    # ── RIGHT PANEL ───────────────────────────────────────────
    with col_R:
        if "result" in st.session_state:
            res=st.session_state["result"]
            tabs=st.tabs(["📊 Visuals","📝 Report","🔬 Model Details","⚙️ Patent Features"])

            with tabs[0]:
                if res["no_tumor"]:
                    st.subheader("Uploaded Scan")
                    st.image(cv2.cvtColor(res['orig'],cv2.COLOR_BGR2RGB),
                             caption="Original MRI — No abnormality detected",
                             use_container_width=True)
                    st.info("ℹ️ No tumor detected — segmentation and GradCAM are not generated for normal scans.")
                else:
                    c1,c2=st.columns(2)
                    with c1:
                        st.image(cv2.cvtColor(res['orig'],cv2.COLOR_BGR2RGB),
                                 caption="Original MRI",use_container_width=True)
                        st.image(res['hmap_r'],caption="Grad-CAM Heatmap",
                                 use_container_width=True,clamp=True)
                    with c2:
                        st.image(cv2.cvtColor(res['mask_ov'],cv2.COLOR_BGR2RGB),
                                 caption=f"Segmentation  |  Ø {res['size_info']['diameter_cm']} cm",
                                 use_container_width=True)
                        st.image(cv2.cvtColor(res['gcam_ov'],cv2.COLOR_BGR2RGB),
                                 caption="Grad-CAM Overlay",use_container_width=True)
                    # Overlap metrics
                    if res.get("overlap"):
                        ov=res["overlap"]
                        oc1,oc2,oc3=st.columns(3)
                        oc1.metric("Overlap (IoU)",str(ov["overlap_score"]))
                        oc2.metric("Attn. in Lesion",f"{ov['attention_inside_lesion_percent']}%")
                        oc3.metric("Exp. Consistency",str(ov["explainability_consistency"]))

            with tabs[1]:
                with st.expander("View Full Radiology Report",expanded=True):
                    for line in res['llm'].split('\n'):
                        line=line.strip()
                        if not line: continue
                        if any(line.startswith(h) for h in
                               ["1.","2.","3.","4.","5.","6."]):
                            st.markdown(f"**{line}**")
                        else:
                            st.write(line)
                if not res['no_tumor'] and res.get('rano'):
                    st.markdown("---"); st.subheader("🏥 RANO Assessment")
                    c1,c2=st.columns(2)
                    with c1:
                        st.metric("Size Category",res['rano']['size_cat'])
                        st.metric("Enhancement",res['rano']['enhancement'])
                    with c2:
                        st.metric("Necrosis",res['rano']['necrosis'])
                    st.caption(res['rano']['grade'])

            with tabs[2]:
                if res.get("model_scores"):
                    st.subheader("Per-Model Class Scores")
                    rows=[]
                    for mn,scores in res["model_scores"].items():
                        row={"Model":mn}; row.update(scores); rows.append(row)
                    st.dataframe(pd.DataFrame(rows),use_container_width=True,hide_index=True)
                if res.get("fusion"):
                    f=res["fusion"]
                    st.subheader("Fusion Summary")
                    st.write({"Agreement":f["agreement_score"],
                              "Uncertainty Flag":f["uncertainty_flag"],
                              "Decision Logic":f["decision_logic"],
                              "Fused Class Scores":f["class_scores"]})

            with tabs[3]:
                feature_rows=[
                    {"#":"1","Feature":"Scan Quality Validation",
                     "Value":res.get("quality",{}).get("quality_score","N/A")},
                    {"#":"2","Feature":"Multi-Model Fusion",
                     "Value":res.get("fusion",{}).get("decision_logic","N/A")},
                    {"#":"3","Feature":"Confidence Decision Logic",
                     "Value":res.get("fusion",{}).get("uncertainty_flag","N/A")},
                    {"#":"4","Feature":"Tumor Segmentation",
                     "Value":("Yes" if not res["no_tumor"] else "N/A (no tumor)")},
                    {"#":"5","Feature":"Tumor Size Measurement",
                     "Value":(f"{res['size_info']['area_cm2']} cm2" if not res["no_tumor"] else "N/A")},
                    {"#":"6","Feature":"Shape Analysis",
                     "Value":(str(res['shape_info']['irregularity']) if (not res["no_tumor"] and res.get("shape_info")) else "N/A")},
                    {"#":"7","Feature":"Explainability Overlay + Overlap",
                     "Value":(str(res.get("overlap",{}).get("explainability_consistency","N/A")))},
                    {"#":"8","Feature":"Reliability Score",
                     "Value":(res['risk_info']['reliability_score'] if not res["no_tumor"] else "N/A")},
                    {"#":"9","Feature":"Severity & Risk Scoring",
                     "Value":f"{res['severity']} / {(res.get('risk_info',{}).get('risk','None'))}"},
                    {"#":"10","Feature":"Prior-Case Comparison",
                     "Value":(res.get("comparison",{}).get("progression_flag","N/A"))},
                    {"#":"11","Feature":"Groq Report Generation","Value":"Enabled" if GROQ_API_KEY else "Key missing"},
                    {"#":"12","Feature":"Doctor Dashboard History","Value":len(load_history())},
                ]
                st.dataframe(pd.DataFrame(feature_rows),use_container_width=True,hide_index=True)
        else:
            st.markdown("""
            <div style='background:#0d0f18;border:1px dashed #1a2a3a;border-radius:14px;
                        padding:60px 40px;text-align:center;margin-top:40px'>
              <div style='font-size:48px;margin-bottom:12px'>🧠</div>
              <div style='font-size:16px;color:#4a6a8a;font-weight:600'>Upload an MRI and click Run Analysis</div>
              <div style='font-size:13px;color:#2a3a50;margin-top:6px'>Results will appear here</div>
            </div>""",unsafe_allow_html=True)

# ════════════════════════════════════════════════════════════════
# PAGE: PATIENT HISTORY
# ════════════════════════════════════════════════════════════════
elif page=="📋 Patient History":
    st.title("📋 Patient History")
    st.caption("All cases analyzed — persisted on Google Drive.")
    st.markdown("---")
    history=load_history()
    if not history:
        st.info("No cases analyzed yet. Go to the Dashboard to analyze an MRI scan.")
    else:
        df=pd.DataFrame(history)
        rename={"id":"Case ID","filename":"File","patient":"Patient","label":"Tumor Type",
                "confidence":"Confidence","severity":"Severity",
                "area_cm2":"Area (cm²)","risk":"Risk","reliability":"Reliability","date":"Date"}
        df=df.rename(columns={k:v for k,v in rename.items() if k in df.columns})
        st.dataframe(df,use_container_width=True,hide_index=True)
        st.markdown("---")
        c1,c2,c3,c4=st.columns(4)
        with c1: st.metric("Total Cases",len(history))
        with c2: st.metric("Severe",sum(1 for h in history if h.get("severity")=="Severe"))
        with c3: st.metric("Tumor Detected",
                            sum(1 for h in history
                                if h.get("label","").lower() not in["no_tumor","no tumor","—"]))
        with c4: st.metric("Normal Scans",
                            sum(1 for h in history
                                if h.get("label","").lower() in["no_tumor","no tumor"]))

# ════════════════════════════════════════════════════════════════
# PAGE: MODEL INFO
# ════════════════════════════════════════════════════════════════
elif page=="ℹ️ Model Info":
    st.title("ℹ️ Model Information"); st.markdown("---")
    st.subheader("Classification Models (Fused)")
    mc1,mc2,mc3=st.columns(3)
    with mc1:
        st.metric("Primary","EfficientNetV2-S")
        st.caption("Weight: 1.2 | Classes: 4 | Input: 384×384")
    with mc2:
        st.metric("Secondary","MobileNetV3")
        st.caption("Weight: 1.0 | Classes: 4 | Input: 384×384")
    with mc3:
        st.metric("Tertiary","ConvNeXt Tiny")
        st.caption("Weight: 1.1 | Classes: 4 | Input: 384×384")
    st.markdown("---")
    c1,c2=st.columns(2)
    with c1:
        st.subheader("Segmentation")
        st.metric("Architecture","EfficientNet U-Net")
        st.metric("Input",f"{SEG_SIZE}×{SEG_SIZE}"); st.metric("Output","Binary Mask")
        st.caption("Pixel-level tumor region segmentation")
    with c2:
        st.subheader("Report Generation")
        st.metric("Explainability","Grad-CAM")
        st.metric("LLM","Llama-4 Scout (Groq)"); st.metric("PDF","2-3 Pages")
        st.caption("Full radiology report with risk analytics")
    st.markdown("---"); st.subheader("MRI Acquisition Parameters")
    for k,v in MRI_PARAMS.items():
        st.write(f"**{k}:** {v}")
    st.markdown("---")
    st.warning("⚠️ This system is for research and screening purposes only. "
               "All AI outputs must be reviewed by a qualified radiologist or neurosurgeon "
               "before any clinical decisions are made.")

'''

# Sanitise unicode dashes that may creep in
APP = (APP
    .replace('\u2014', '-')
    .replace('\u2013', '-')
    .replace('\u2012', '-'))

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(APP)
print(f'✅ app.py written — {APP.count(chr(10))} lines')


✅ app.py written — 1432 lines


In [10]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Launch Streamlit + ngrok                              ║
# ╚══════════════════════════════════════════════════════════════════╝
import subprocess, time, requests
from pyngrok import ngrok, conf

# Kill stale processes
subprocess.run(["pkill", "-9", "-f", "streamlit"], capture_output=True)
subprocess.run(["pkill", "-9", "-f", "ngrok"],     capture_output=True)
time.sleep(2)

# Start Streamlit
proc = subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
])

print("Waiting for Streamlit to start…")
for i in range(30):
    try:
        if requests.get("http://localhost:8501").status_code == 200:
            print("✅ Streamlit is up!"); break
    except:
        pass
    time.sleep(2)
    print(f"  …{i*2}s")

# Tunnel via ngrok
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(8501)

print("\n" + "=" * 60)
print("🚀  NeuroScan AI v5 is LIVE!")
print(f"🌐  Public URL: {public_url}")
print("Keep this Colab tab open while using the app.")
print("=" * 60)


Waiting for Streamlit to start…
  …0s
  …2s
  …4s
  …6s
✅ Streamlit is up!

🚀  NeuroScan AI v5 is LIVE!
🌐  Public URL: NgrokTunnel: "https://trachytic-unabjured-fernanda.ngrok-free.dev" -> "http://localhost:8501"
Keep this Colab tab open while using the app.
